In [0]:

# CÉLULA 1 — SETUP
# DQ MONITORING LOGS - ECOMMERCE_CLIENTES
# ============================================================

import os
import uuid
from datetime import datetime, timezone

from dotenv import load_dotenv
from pyspark.sql import functions as F
import pandas as pd
from deltalake import DeltaTable, write_deltalake

load_dotenv(".env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

STORAGE_ACCOUNT = "internshipdatalake"
SQUAD_CONTAINER = "squad1"

lakehouse_base = (
    f"abfss://{SQUAD_CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net"
)

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

# Caminho da nossa Silver
silver_clientes_path = (
    f"{lakehouse_base}/silver/ecommerce_clientes"
)

# Log compartilhado da Squad 1
dq_logs_path = (
    f"{lakehouse_base}/dq_monitoring_logs"
)

def ler_delta_direto(caminho):
    dt = DeltaTable(
        caminho,
        storage_options=storage_options
    )

    return spark.createDataFrame(dt.to_pandas())

print("✅ Setup concluído.")
print("Silver:", silver_clientes_path)
print("DQ Logs:", dq_logs_path)

In [0]:
# ============================================================
# CÉLULA 2 — REGRAS DE QUALIDADE
# ecommerce_clientes
# ============================================================

from pyspark.sql import functions as F

# 1. Ler a Silver ecommerce_clientes
df_silver = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(silver_clientes_path)
)

print("✅ Silver ecommerce_clientes carregada")

# 2. Regras de qualidade existentes na Silver
REGRAS_CLIENTES = {
    "status_uuid_cliente": "Crítica",
    "status_email": "Crítica",
    "status_senha_hash": "Crítica",
    "status_dt_cadastro": "Crítica"
}

# 3. Verificar quais regras realmente existem
regras_encontradas = [
    regra
    for regra in REGRAS_CLIENTES
    if regra in df_silver.columns
]

print("\n===== REGRAS DQ - ECOMMERCE_CLIENTES =====")
print("Regras encontradas:", regras_encontradas)

for regra in regras_encontradas:
    print(
        f"{regra} -> severidade: "
        f"{REGRAS_CLIENTES[regra]}"
    )

In [0]:
# ============================================================
# CÉLULA 3 — GERAR LOGS DQ
# ecommerce_clientes
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_id = str(uuid.uuid4())
agora = datetime.now(timezone.utc)

total = df_silver.count()

# Definir quais valores representam falha
VALORES_FALHA = {
    "status_uuid_cliente": ["INVÁLIDO"],
    "status_email": ["INVÁLIDO"],
    "status_senha_hash": ["INVÁLIDO"],
    "status_dt_cadastro": ["CADASTRO_FUTURO"]
}

linhas_logs = []

for regra in regras_encontradas:

    valores_falha = VALORES_FALHA[regra]

    qtd_falhas = (
        df_silver
        .filter(F.col(regra).isin(valores_falha))
        .count()
    )

    status = "FAIL" if qtd_falhas > 0 else "PASS"

    linhas_logs.append(
        (
            run_id,
            "ecommerce_clientes",
            regra,
            status,
            REGRAS_CLIENTES[regra],
            qtd_falhas,
            total,
            agora
        )
    )

print("===== RESULTADO DQ =====")

for linha in linhas_logs:
    print(
        linha[2],
        "|", linha[3],
        "| falhas:", linha[5],
        "| total:", linha[6]
    )

In [0]:
# ============================================================
# CÉLULA 4 — GRAVAR DQ_MONITORING_LOGS
# ============================================================

from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, TimestampType
)

# Schema explícito dos logs
schema_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", LongType(), False),
    StructField("qtd_registros_total", LongType(), False),
    StructField("timestamp_execucao", TimestampType(), False)
])

# Criar DataFrame com os resultados da célula 3
df_novo = spark.createDataFrame(
    linhas_logs,
    schema=schema_logs
)

print("===== LOGS QUE SERÃO GRAVADOS =====")
display(df_novo)

In [0]:
# ============================================================
# CÉLULA 5 — SALVAR NO DQ_MONITORING_LOGS
# ============================================================

print("Salvando logs de ecommerce_clientes...")

(
    df_novo.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .save(dq_logs_path)
)

print("✅ Logs de ecommerce_clientes gravados com sucesso!")

In [0]:
# ============================================================
# CÉLULA 6 — VALIDAÇÃO FINAL DO DQ_MONITORING_LOGS
# ============================================================

df_logs_final = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(dq_logs_path)
)

df_clientes_logs = (
    df_logs_final
    .filter(F.col("tabela") == "ecommerce_clientes")
    .orderBy(F.col("timestamp_execucao").desc())
)

print("===== DQ MONITORING - ECOMMERCE_CLIENTES =====")

display(
    df_clientes_logs.select(
        "run_id",
        "tabela",
        "regra",
        "status",
        "severidade",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao"
    )
)